# AIRPATH-AI — Milestone 1 data audit

This notebook audits the supplied HealthyAir station observations. It does not perform interpolation, forecasting, or route optimization. Suspicious PM2.5 observations are flagged and retained.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loading import column_schema, load_air_quality_csv
from src.data_validation import audit_dataset, write_outputs

DATA_PATH = ROOT / "data" / "raw" / "Air Quality Ho Chi Minh City.csv"

In [ ]:
raw = load_air_quality_csv(DATA_PATH)
print(f"Rows: {len(raw):,}; columns: {len(raw.columns)}")
display(column_schema(raw))
display(raw.dtypes.rename("source_dtype").to_frame())

In [ ]:
clean, audit = audit_dataset(raw)
display(audit["parse_issues"])
display(audit["missingness_by_variable"])
display(audit["pm25_statistics"])
display(audit["coverage"])
display(audit["correlations"])
display(audit["split"])

In [ ]:
write_outputs(clean, audit, ROOT)
print("Audit outputs written under data/processed and reports.")

## Leakage safeguards for later milestones

- Construct lags using only information available at prediction time.
- Never include future PM2.5 in features.
- Fit preprocessing parameters on training data only.
- Keep chronological train/validation/test ordering; do not randomly shuffle the time series.
- Initial forecast horizons are t+1, t+2, and t+3 hours. Do not manufacture sub-hourly ground truth.